# 40. The load that never updates anything

**Track:** debug-the-ai | **Difficulty:** medium | **Tags:** upsert, merge, incremental load, left anti join, deduplication

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("040-load-never-updates")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "batch_orders": """
UEFSMRUEFYAIFaoETBWAARUAEgAAgAQEKwAJAQAwCQcIAGkACQEAGwkHBABsDQgAQw0IAG4NCAB4DQgALg0IAA4NCABoDQgARQ0I
AHUNCAB3DQgAcA0IBHQACQEAZwkHCAA6AAkBAHIJBwQAFw0IADYNCAAYDQgAUg0IAFYNCABrDQgATA0IAHENCABgDQgAIA0IABMN
CABNDQgAPA0IAGoNCAASDQgAYg0IAAsNCAAvDQgAbQ0IAFENCABvDQgEHQAJAQBJCQcEAGQNCAAjDQgAZg0IAFMNCAAzDQgAdg0I
AFgNCABeDQgAcw0IAAkNCABADQgADw0IAB4NCAAtDQgAKg0IABENCABVDQgAJA0IAD0NCABLDQg8ZQAAAAAAAAAKAAAAAAAAABUA
FX4VhAEsFYoBFRAVBhUGHBgIeAAAAAAAAAAYCAkAAAAAAAAAFgAoCHgAAAAAAAAAGAgJAAAAAAAAABERAAAAP/A+AwAAAIoBAQYT
QCAMRGEcSKIsTOMMDxRJE1VZF5ZpG2d13geG4kiW5omm6sq27gvHcjDTX23feK7vfO+3PwAAFQQVchVuTBUKFQASAAA5fAcAAABT
SElQUEVECQAAAENBTkNFTExFRAYAAABQQUNLDQpETEFDRUQJAAAAREVMSVZFUkVEFQAVSBVMLBWKARUQFQYVBhw2ACgHU0hJUFBF
RBgJQ0FOQ0VMTEVEEREAAAAkjAMAAACKAQEDEwAiSdgADBs0DIM0UpCUcNi0DRohUSRELAkoABUEFYoFFZQFTBWCARUAEgAAxQL0
RAEAAAlJfQAAAtIrAAADrN0AAAp54wAABFK3AAABO2IAAA2TIgAAAlgkAAAEk2IAAATASQAABKmpAAAIFL0AAAKwpgAACSVEAAAK
f70AAArbiwAAAYJSAAAMLxAAAAx/HwAACSHOAAALlA0AAAjljgAACuKoAAALoXEAAAgSYQAABIdnAAAKtAYAAALj9gAABEifAAAI
2fYAAAen7QAACJYdAAAKh0MAAAOQDwAABImaAAAHDiYAAAffvQAACDOQAAAByC0AAAPUIgAAAFylAAALBykAAAoCuwAAB2UwAAAL
Qk4AAASJYAAAA/feAAAJ1uIAAA2ooAAAB94vAAAB4T4AAAlHmQAAC8LwAAAJ42YAAAyFZQAACz3vAAAJpEYAAALrtQAABs9tAAAE
HP0AAAP4TgAAA3zdAAAFSuoAAAH2agAACckgFQAVkAEVlgEsFYoBFRAVBhUGHBgFAAANqKAYBQAAAFylFgAoBQAADaigGAUAAABc
pRERAAAASPBHAwAAAIoBAQcTgIBgQCgYDoiEYsFoOB6QiGRCqVgumIxmw+lcPB9QSDQilUwnVEq1YrVcL1hMNgvQaiDbDZfT7Xg9
388FBAAAFQQV0AgVnghMFYoBFQASAACoBGyAlRmw6j4GAAA4RUo5PwYAQD8eEVw/BgCA0aJzARAQAIRShmcBEAxi19tgAQgQmJgn
QT8BKAgJ0jIBCCytdQFlPwYAANrh5lQBSAjJ3nYBIEyAO87/dz4GAMBpTpFRPwYAQBNiTgVQ8EMUzhFsPwYAgL1d9zs/BgDAxN3y
aT8GAABAkwhSPwYAgNTLt5I+BgCA+n5WWz8GAEAjNBV4PgYAgG6aPqg+BgAALqm9ngFYKKkRs0A/BgAAFu4FAYB0QFRuHU8/BgCA
IrnAYT8GAECrMIJdPwYAAJBk4JU+AaAoABVDPwYAwCUbkUYBCAzQebc+AQgMKw4DXgEICF5v8wFw8EPAXZk6Wj8GAIAj5hY9PwYA
APrXikQ/BgDAkvyrZD8GAEB5V71WPgYAQEId+1Y/BgAAJXrbaD4GAMDhPaBEPwYAgNYjSAGoMED6uMcxPwYAQOpVde0BIIg2uUW2
PgYAgBD9Tzc/BgCAQ8dgTD8GAECqH1/ZPgYAwAptzQGYUEC7balIPwYAAHlK5nU+BgCAZYLbMAEQKITX608/BgCAOL5NJWAoR+x2
hj4GAMCCjvcBEBAAO9WbRQEgLKo/UAU/BgBA/oqcyAEgCEkC5yEAFMB3/jV8PgFYRGQPpj4GAMBWEw48PwYAgP6n2gE4EACsbd02
ARAIaTOTIZA8gK8NEjQ/BgAAJ+NyZT8GABUAFZABFZYBLBWKARUQFQYVBhwYCAAUzhFsPwYAGAgA2uHmVD4GABYAKAgAFM4RbD8G
ABgIANrh5lQ+BgAREQAAAEjwRwMAAACKAQEHE4CAYEAoGA6IhGLBaDgekIhkQqlYLpiMZsPpeD6gkGhEKplOqJRqxWq5XrCYbEar
2W64nG7H6/l+wKBwSAQAABUEGVw1ABgGc2NoZW1hFQgAFQQlAhgIb3JkZXJfaWQAFQwlAhgGc3RhdHVzJQBMHAAAABUOFQoVAhgG
YW1vdW50JQoVBBUULFwVBBUUAAAAFQQlAhgKdXBkYXRlZF9hdCUUTIwSHCwAAAAAABaKARkcGUwmABwVBBk1AAYQGRgIb3JkZXJf
aWQVAhaKARaiChbSBibUBCYIHBgIeAAAAAAAAAAYCAkAAAAAAAAAFgAoCHgAAAAAAAAAGAgJAAAAAAAAABERABksFQQVABUCABUA
FRAVAgA8KQYZJgCKAQAAACYAHBUMGTUABhAZGAZzdGF0dXMVAhaKARauAhauAibkBybaBhw2ACgHU0hJUFBFRBgJQ0FOQ0VMTEVE
EREAGSwVBBUAFQIAFQAVEBUCADwWvAcZBhkmAIoBAAAAJgAcFQ4ZNQAGEBkYBmFtb3VudBUCFooBFqgHFrgHJr4OJogJHBgFAAAN
qKAYBQAAAFylFgAoBQAADaigGAUAAABcpRERABksFQQVABUCABUAFRAVAgA8KQYZJgCKAQAAACYAHBUEGTUABhAZGAp1cGRhdGVk
X2F0FQIWigEWhgsW2gomgBkmwBAcGAgAFM4RbD8GABgIANrh5lQ+BgAWACgIABTOEWw/BgAYCADa4eZUPgYAEREAGSwVBBUAFQIA
FQAVEBUCADwpBhkmAIoBAAAAFv4eFooBJggWkhsAGRwYDEFSUk9XOnNjaGVtYRiYAy8vLy8veWdCQUFBUUFBQUFBQUFLQUF3QUJn
QUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFRQUFBQzhBQUFBZkFBQUFFQUFBQUFFQUFBQVpQLy8v
d0FBQVFvUUFBQUFKQUFBQUFRQUFBQUFBQUFBQ2dBQUFIVndaR0YwWldSZllYUUFBQUFBQmdBSUFBWUFCZ0FBQUFBQUFnQ2MvLy8v
QUFBQkJ4QUFBQUFnQUFBQUJBQUFBQUFBQUFBR0FBQUFZVzF2ZFc1MEFBQUlBQXdBQkFBSUFBZ0FBQUFLQUFBQUFnQUFBTlQvLy84
QUFBRUZFQUFBQUJ3QUFBQUVBQUFBQUFBQUFBWUFBQUJ6ZEdGMGRYTUFBQVFBQkFBRUFBQUFFQUFVQUFnQUJnQUhBQXdBQUFBUUFC
QUFBQUFBQUFFQ0VBQUFBQ1FBQUFBRUFBQUFBQUFBQUFnQUFBQnZjbVJsY2w5cFpBQUFBQUFJQUF3QUNBQUhBQWdBQUFBQUFBQUJR
QUFBQUE9PQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMjUuMC4xGUwcAAAcAAAcAAAcAAAA+AMAAFBBUjE=
""",
    "target_orders": """
UEFSMRUEFcAMFb4GTBXIARUAEgAAoAYELgAJAQAbCQcEAEkNCAAqDQgAKQ0IAEQNCABDDQgAYw0IAEsNCABdDQgAMw0IAAcNCAAC
DQgEOgAJAQBaCQcEAEUNCAANDQgAXg0IABQNCAAFDQgAYA0IADENCAAYDQgALw0IACYNCABMDQgATg0IAEgNCAADDQgADw0IAD8N
CAAjDQgARg0IABINCAA4DQgAEQ0IACwNCABZDQgAYQ0IAFwNCAAwDQgAHA0IACENCABYDQgAQQ0IABUNCAAnDQgAOQ0IADwNCABb
DQgANQ0IAAkNCABNDQgACA0IAGQNCAAQDQgAJA0IAFMNCAAeDQgAIg0IAC0NCAA+DQgARw0IABcNCABWDQgANw0IAA4NCAABDQgA
VA0IAFENCAA0DQgAQA0IADINCAAZDQgASg0IADYNCAA7DQgACg0IAE8NCAA9DQgAIA0IAAYNCAAEDQgAEw0IABoNCAAdDQgAKA0I
AFcNCAAfDQgADA0IACsNCABQDQgAUg0IAGINCABCDQgAJQ0IAAsNCABVDQg8FgAAAAAAAABfAAAAAAAAABUAFcgBFc4BLBXIARUQ
FQYVBhwYCGQAAAAAAAAAGAgBAAAAAAAAABYAKAhkAAAAAAAAABgIAQAAAAAAAAAREQAAAGTwYwMAAADIAQEHG4CAYEAoGA6IhGLB
aDgekIhkQqlYLpiMZsPpeD6gkGhEKplOqJRqxWq5XrCYbEar2W64nG7H6/l+wKBwSCwaj8ikcslsOp/QqHRKrVqv2Kx2y+16v+Cw
eAwAAAAVBBU+FUJMFQYVABIAAB94BgAAAFBBQ0tFRAcAAABTSElQUEVEBgAAAFBMQUNFRBUAFUYVSiwVyAEVEBUGFQYcNgAoB1NI
SVBQRUQYBlBBQ0tFRBERAAAAI4gDAAAAyAEBAhsQJYVlpmZmmoUgahSkpBqgRZZhZVpaUgoWABUEFegHFfIHTBXIARUAEgAA9AP0
8wEAAAPQdgAACnnjAAAKArsAAAKKDQAAB6WVAAABkLYAAAE7YgAAB7UMAAAFSuoAAAEsAgAACdbiAAAEBxoAAAS0pQAADH8fAAAB
0HsAAAgUvQAAC3m0AAAB4T4AAAZcmgAACcDIAAAESJ8AAALdaQAACuKoAAAIM5AAAAOuTwAACrQGAAAERYsAAALyfAAAAomsAAAM
hWUAAAzDYQAAC0JOAAAMjvsAAASJmgAAB0LsAAAGDIEAAAxB5AAADTmhAAAI4sAAAAbh/wAAAg8/AAAGOmIAAAezigAAB94vAAAA
neYAAAT++wAABVcLAAAAW2oAAAqHQwAABzO1AAALMJAAAAvC8AAACJYdAAAAutAAAAdlMAAAAcoDAAAD+E4AAAP33gAACz3vAAAK
i5AAAAmkRgAADIFeAAAK3HYAAAuUDQAACBJhAAAFv3sAAATASQAAAhYjAAAFl48AAAPUIgAABFPqAAAJ42YAAAFSVQAAAZTzAAAI
7xQAAAjljgAAAgH7AAAJBjQAAAW27AAAA3zdAAAI2fYAAADILAAADCfmAAAHp+0AAAZjHwAACwcpAAAISvkAAAJsEwAACg0gAAAN
QY0AAAlJfQAACSr8AAALoXEAAAcOJgAADZygAAAGuUsAAAd+FQAABBz9AAAGF2EAAAv04xUAFcgBFc4BLBXIARUQFQYVBhwYBQAA
DZygGAUAAABbahYAKAUAAA2coBgFAAAAW2oREQAAAGTwYwMAAADIAQEHG4CAYEAoGA6IhGLBaDgekIhkQqlYLpiMZsPpeD6gkGhE
KplOqJRqxWq5XrCYbEar2W64nG7H6/l+wKBwSCwaj8ikcslsOp/QqHRKrVqv2Kx2y+16v+CweAwAAAAVBBXADBXoC0wVyAEVABIA
AKAGcMB8xd+IPgYAgKoNLMA+BgAAgylozj4GAMCJPFCjAQhsUn37iT4GAIDBYwN8PgYAAL+F6KE+BgBAV/UWpAEIDDsIg8gBCExd
+6IGPwYAQKofX9k+BgDA+bx64gEIiIrB46I+BgCA1Mu3kj4GAMDvNsR+PgYAgDvO/3c+BgDAb3BHAXAQAHlK5nUBECjWAld6PgYA
QI4qHAUYKJBk4JU+BgCA/gErBRBMLqm9nj4GAEDx3kt9PgYAwMDJkvEBCCwPy0TCPgYAAK+g9Z0BCEyjGNCnPgYAQLbyH4A+BgAA
7ChHuQEITJ0nleg+BgDANrlFtj4GAEAhEM6FAQjIkLS1cT4GAAB05LaNPgYAwIM/E7s+BgCA6aHS9T4GAECWSXvKPgYAQIthoNQ+
BgCAdwwfAehUQLSYaow+BgAADHXWvT4GAMCt+L3DPgEwCGFtogEIaBe+aYI+BgAAagKUAD8GAMAwNgObPgYAwAPZZwGgEMA++wrj
ARAI0pmZAUDwPACVhxGDPgYAQHXSwwU/BgBABOlT1T4GAABP3h/wPgYAQOpVde0+BgCAx3nbwz4GAMB0SvBuPgYAgIiOjncBCAiq
P1ABOCwAuWgT7z4GAED+ipwhoBAAtdiKjgEQDFIqI9sBCCgjNBV4PgYAwAbTZwGQLIBuA1aCPgYAAMaFDyEALECeMDVtPgYAwKSf
ZgEoEAAlettoARAoJqcq7j4GAIDLyz0lQAwi+AnFARAMxG7vnwEIDC79NZgBCAxumj6oAQgMIkbD/wEIRKoFhs0+BgAAurI8bD4G
AMAveiU4LICLsN6iPgYAgN35+SHQbIB4cz/sPgYAwH5CBOo+BgAArWH3qj4GAIDWI0gFUAyEC7HHARAMGz+E8gEIKFMkIAI/BgCA
QzSwAaAMgJUZsAU4CJsuSAGQbMDd9y0BPwYAQCYN5pM+BgDAqqItsj4GAIDe0ncBUHxAyhz39j4GAABR2harPgYAQP+HDvQ+BgAA
4m/wuT4GABUAFcgBFc4BLBXIARUQFQYVBhwYCEBd+6IGPwYAGAiAQzSwaD4GABYAKAhAXfuiBj8GABgIgEM0sGg+BgAREQAAAGTw
YwMAAADIAQEHG4CAYEAoGA6IhGLBaDgekIhkQqlYLpiMZsPpeD6gkGhEKplOqJRqxWq5XrCYbEar2W64nG7H6/l+wKBwSCwaj8ik
cslsOp/QqHRKrVqv2Kx2y+16v+CweAwAAAAVBBlcNQAYBnNjaGVtYRUIABUEJQIYCG9yZGVyX2lkABUMJQIYBnN0YXR1cyUATBwA
AAAVDhUKFQIYBmFtb3VudCUKFQQVFCxcFQQVFAAAABUEJQIYCnVwZGF0ZWRfYXQlFEyMEhwsAAAAAAAWyAEZHBlMJgAcFQQZNQAG
EBkYCG9yZGVyX2lkFQIWyAEWrg8Wsgkm6AYmCBwYCGQAAAAAAAAAGAgBAAAAAAAAABYAKAhkAAAAAAAAABgIAQAAAAAAAAAREQAZ
LBUEFQAVAgAVABUQFQIAPCkGGSYAyAEAAAAmABwVDBk1AAYQGRgGc3RhdHVzFQIWyAEW8gEW+gEmmAomugkcNgAoB1NISVBQRUQY
BlBBQ0tFRBERABksFQQVABUCABUAFRAVAgA8FoAKGQYZJgDIAQAAACYAHBUOGTUABhAZGAZhbW91bnQVAhbIARa+ChbOCibIEya0
CxwYBQAADZygGAUAAABbahYAKAUAAA2coBgFAAAAW2oREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAyAEAAAAmABwVBBk1AAYQGRgK
dXBkYXRlZF9hdBUCFsgBFq4PFtwOJowiJoIWHBgIQF37ogY/BgAYCIBDNLBoPgYAFgAoCEBd+6IGPwYAGAiAQzSwaD4GABERABks
FQQVABUCABUAFRAVAgA8KQYZJgDIAQAAABaMKxbIASYIFtYkABkcGAxBUlJPVzpzY2hlbWEYmAMvLy8vL3lnQkFBQVFBQUFBQUFB
S0FBd0FCZ0FGQUFnQUNnQUFBQUFCQkFBTUFBQUFDQUFJQUFBQUJBQUlBQUFBQkFBQUFBUUFBQUM4QUFBQWZBQUFBRUFBQUFBRUFB
QUFaUC8vL3dBQUFRb1FBQUFBSkFBQUFBUUFBQUFBQUFBQUNnQUFBSFZ3WkdGMFpXUmZZWFFBQUFBQUJnQUlBQVlBQmdBQUFBQUFB
Z0NjLy8vL0FBQUJCeEFBQUFBZ0FBQUFCQUFBQUFBQUFBQUdBQUFBWVcxdmRXNTBBQUFJQUF3QUJBQUlBQWdBQUFBS0FBQUFBZ0FB
QU5ULy8vOEFBQUVGRUFBQUFCd0FBQUFFQUFBQUFBQUFBQVlBQUFCemRHRjBkWE1BQUFRQUJBQUVBQUFBRUFBVUFBZ0FCZ0FIQUF3
QUFBQVFBQkFBQUFBQUFBRUNFQUFBQUNRQUFBQUVBQUFBQUFBQUFBZ0FBQUJ2Y21SbGNsOXBaQUFBQUFBSUFBd0FDQUFIQUFnQUFB
QUFBQUFCUUFBQUFBPT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRlMHAAAHAAAHAAAHAAAAPUDAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "040-load-never-updates"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

batch_orders = spark.read.parquet((_data_dir / "batch_orders.parquet").resolve().as_posix())
batch_orders.createOrReplaceTempView("batch_orders")
target_orders = spark.read.parquet((_data_dir / "target_orders.parquet").resolve().as_posix())
target_orders.createOrReplaceTempView("target_orders")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'amount': '32525ad26faccf3139f5dd4fca4715f8bfd944dd0e324e422882eaf85d914184',
                         'order_id': '9cea9f5eaedb5163a76c9225f5a192e81d64f89252788d7a054eb745de435b7a',
                         'status': '74ab72faba88a99e319776029ece3faa55233ec273e47c7f7d02391b2f6195f4',
                         'updated_at': '1e2358f21e5887ba9f37164a2639b8fe497bd29d2a934a9ac4db8cae6223cbe3'},
 'columns': [['amount', 'decimal(10,2)'],
             ['order_id', 'bigint'],
             ['status', 'string'],
             ['updated_at', 'timestamp_ntz']],
 'fingerprint': '59b6c6c30b7d8c0baa15d7b94899c92b16c990b763a0741f6601029d81155a42',
 'order_matters': False,
 'row_count': 120}

check = make_check(EXPECTED)

## The situation

The orders table is refreshed from batches sent by the order system. A batch contains
every order that was created or changed since the previous one.

An AI assistant wrote the load in the answer cell. It has run for a week. The row
count after each load is exactly what the order system reports, so the check passes.
Customer support now says that orders delivered days ago still show as `PLACED`.

Find the bug and fix it.

## Input tables

### `target_orders`

The table as it is before the load. One row per order.

### `batch_orders`

The incoming batch. It can contain new orders, newer versions of orders the target
already has, the same order more than once, a row the target already has (the order
system re-sends on a timeout), and a version that is **older** than what the target
has (a delayed message).

Both tables have the same columns:

| Column | Type | Description |
|---|---|---|
| `order_id` | BIGINT | The order. Never NULL. |
| `status` | STRING | `PLACED`, `PACKED`, `SHIPPED`, `DELIVERED` or `CANCELLED`. |
| `amount` | DECIMAL(10,2) | The order value. It can change between versions. |
| `updated_at` | TIMESTAMP_NTZ | When this version of the order was saved. |

Two versions of an order with the same `updated_at` are identical in every column.

## Requirement

Return the orders table as it should be after the load.

- One row per `order_id` found in either table.
- For each order, keep the version with the latest `updated_at`, whichever table it
  is in. A batch row that is older than the target's row changes nothing.
- All four columns come from that one version.

## Expected output

One row per order.

| Column | Type | Description |
|---|---|---|
| `order_id` | BIGINT | |
| `status` | STRING | |
| `amount` | DECIMAL(10,2) | |
| `updated_at` | TIMESTAMP_NTZ | |

In [ ]:
batch_orders.show(5)
target_orders.show(5)

## Your answer

The cell below is the AI-written code. It runs, but its output is wrong. Find the bug and fix it. The tables are available as DataFrames and as views: `batch_orders`, `target_orders`.

In [ ]:
def solve(spark):
    """Apply the incoming batch to the orders table."""
    target = spark.table("target_orders")
    batch = spark.table("batch_orders")

    # Orders in the batch that the target does not have yet
    new_orders = batch.join(target, on="order_id", how="left_anti")

    # Add them to the existing orders
    result = target.unionByName(new_orders)

    return result


result = solve(spark)
result.show()

In [ ]:
check(result)